# TMDB Title Matching and Metadata Enrichment

This notebook matches IMDb title strings to TMDB movie and TV records, evaluates match confidence, preserves trusted matches, and retrieves detailed TMDB metadata for downstream analysis.

## Workflow

1. Load IMDb title candidates.
2. Search TMDB for movie and TV matches.
3. Score candidate matches using title and year agreement.
4. Validate the matching logic on a pilot sample.
5. Run matching across all candidates.
6. Retain high-confidence matches.
7. Retrieve detailed TMDB metadata.
8. Validate and deduplicate the final metadata table.

## Setup

Import the required libraries, define project paths, and load the TMDB API credentials from the local environment file.

In [1]:
from pathlib import Path
import os
import re

import pandas as pd
import requests
from dotenv import load_dotenv

# Get the main project folder from the notebooks folder
project_root = Path.cwd().parent

# Load the TMDB API token from the project's .env file
load_dotenv(project_root / ".env")

tmdb_token = os.getenv("TMDB_READ_TOKEN")

tmdb_base_url = "https://api.themoviedb.org/3"

headers = {
    "Authorization": f"Bearer {tmdb_token}",
    "accept": "application/json"
}

## 1. Load IMDb Title Candidates

The IMDb review corpus was previously profiled and filtered to titles with at least 100 reviews. Episode-like titles were excluded before this matching stage.

In [2]:
# Load the IMDb titles selected for TMDB enrichment
candidate_file = (
    project_root
    / "data"
    / "processed"
    / "imdb_tmdb_candidates.parquet"
)

tmdb_candidates = pd.read_parquet(candidate_file)

print(f"Candidates loaded: {len(tmdb_candidates):,}")

tmdb_candidates.head()

Candidates loaded: 8,841


,movie,review_count,is_episode,is_video,year,search_title
0,Avengers: Endgame (2019),8771,False,False,2019,Avengers: Endgame
1,The Shawshank Redemption (1994),8236,False,False,1994,The Shawshank Redemption
2,Dil Bechara (2020),7764,False,False,2020,Dil Bechara
3,Captain Marvel (2019),7158,False,False,2019,Captain Marvel
4,The Dark Knight (2008),6828,False,False,2008,The Dark Knight


## 2. Verify TMDB Connection

Confirm that the TMDB API credentials and connection are working before starting the matching process.

In [35]:
# Confirm that the TMDB API connection is working
test_response = requests.get(
    f"{tmdb_base_url}/search/multi",
    headers=headers,
    params={
        "query": "Game of Thrones",
        "include_adult": False
    },
    timeout=30
)

print("Status code:", test_response.status_code)

Status code: 200


## 3. Define TMDB Matching Logic

TMDB search results are scored using title agreement, original-title agreement, release year, and search-result rank. A combined movie/TV search is attempted first, followed by year-constrained movie and TV searches when needed.

In [3]:
def get_result_year(result):
    # Movies and TV shows use different date fields
    if result.get("media_type") == "movie":
        date = result.get("release_date")
    else:
        date = result.get("first_air_date")

    # Return only the four-digit year when a date is available
    if date:
        return date[:4]

    return None


def normalize_title(title):
    # Make title comparisons less sensitive to capitalization and extra spaces
    if not title:
        return ""

    title = title.casefold()
    title = re.sub(r"\s+", " ", title)

    return title.strip()


def score_tmdb_result(result, search_title, search_year, rank):
    # Get the title fields based on whether this is a movie or TV result
    if result.get("media_type") == "movie":
        result_title = result.get("title", "")
        original_title = result.get("original_title", "")
    else:
        result_title = result.get("name", "")
        original_title = result.get("original_name", "")

    result_year = get_result_year(result)

    search_title_normalized = normalize_title(search_title)
    result_title_normalized = normalize_title(result_title)
    original_title_normalized = normalize_title(original_title)

    score = 0

    # Give a strong score when the returned title exactly matches the IMDb title
    if search_title_normalized == result_title_normalized:
        score += 5

    # Also check the original TMDB title
    if search_title_normalized == original_title_normalized:
        score += 3

    # The release or first-air year is one of our strongest matching signals
    if result_year == str(search_year):
        score += 5

    # TMDB already ranks search results by relevance, so use rank as a small tie-breaker
    score += max(0, 2 - (rank * 0.2))

    return score

In [4]:
def search_tmdb_candidate(search_title, search_year):
    # Try one combined movie/TV search first
    multi_response = requests.get(
        f"{tmdb_base_url}/search/multi",
        headers=headers,
        params={
            "query": search_title,
            "include_adult": False
        },
        timeout=30
    )

    multi_response.raise_for_status()

    multi_results = [
        result
        for result in multi_response.json().get("results", [])
        if result.get("media_type") in ["movie", "tv"]
    ]

    scored_results = []

    # Score the results from the first search
    for rank, result in enumerate(multi_results):
        score = score_tmdb_result(
            result,
            search_title,
            search_year,
            rank
        )

        scored_results.append((score, result))

    # Use the multi-search result if it has both a strong score and the correct year
    if scored_results:
        scored_results.sort(key=lambda x: x[0], reverse=True)

        best_score, best_result = scored_results[0]
        best_year = get_result_year(best_result)

        if best_score >= 7 and best_year == str(search_year):
            return best_result, best_score, "multi"

    # If multi-search is not convincing, search movies with the year included
    movie_response = requests.get(
        f"{tmdb_base_url}/search/movie",
        headers=headers,
        params={
            "query": search_title,
            "year": search_year,
            "include_adult": False
        },
        timeout=30
    )

    movie_response.raise_for_status()

    movie_results = movie_response.json().get("results", [])

    # Add media_type so movie and TV results have the same structure
    for result in movie_results:
        result["media_type"] = "movie"

    # Also search TV with the first-air year included
    tv_response = requests.get(
        f"{tmdb_base_url}/search/tv",
        headers=headers,
        params={
            "query": search_title,
            "first_air_date_year": search_year,
            "include_adult": False
        },
        timeout=30
    )

    tv_response.raise_for_status()

    tv_results = tv_response.json().get("results", [])

    for result in tv_results:
        result["media_type"] = "tv"

    fallback_results = movie_results + tv_results

    if not fallback_results:
        return None, None, "no_match"

    scored_results = []

    for rank, result in enumerate(fallback_results):
        score = score_tmdb_result(
            result,
            search_title,
            search_year,
            rank
        )

        scored_results.append((score, result))

    scored_results.sort(key=lambda x: x[0], reverse=True)

    best_score, best_result = scored_results[0]

    return best_result, best_score, "fallback"

In [5]:
def assign_match_status(result, score, search_method):
    # Keep API errors separate so we know they need to be retried
    if search_method == "request_error":
        return "request_error"

    # No TMDB result was found
    if result is None:
        return "no_match"

    # Strong matches can be accepted automatically
    if score >= 12:
        return "high_confidence"

    # Weaker matches are kept but flagged so we can review them later
    return "low_confidence"

### Quick Matching Check

Test the matching function on a TV show, a movie, and a non-English title before applying it to the IMDb candidate data.

In [17]:
tests = [
    ("Game of Thrones", 2011),
    ("Avengers: Endgame", 2019),
    ("小丑", 2019)
]

for title, year in tests:
    result, score, search_method = search_tmdb_candidate(title, year)

    print(f"\nIMDb: {title} ({year})")
    print("Search method:", search_method)
    print("Score:", score)

    if result:
        print("TMDB ID:", result.get("id"))
        print("Media type:", result.get("media_type"))
        print(
            "TMDB title:",
            result.get("title") or result.get("name")
        )
        print("TMDB year:", get_result_year(result))


IMDb: Game of Thrones (2011)
Search method: multi
Score: 15.0
TMDB ID: 1399
Media type: tv
TMDB title: Game of Thrones
TMDB year: 2011

IMDb: Avengers: Endgame (2019)
Search method: multi
Score: 15.0
TMDB ID: 299534
Media type: movie
TMDB title: Avengers: Endgame
TMDB year: 2019

IMDb: 小丑 (2019)
Search method: fallback
Score: 7.0
TMDB ID: 475557
Media type: movie
TMDB title: Joker
TMDB year: 2019


## 4. Validate Matching Logic on a Pilot Sample

Before running thousands of TMDB searches, apply the matching process to the first 100 IMDb candidates. Weak or unresolved matches and suspicious year differences are reviewed before scaling the process.

In [6]:
# Use the first 100 candidates to validate the matching process
pilot_candidates = tmdb_candidates.head(100).copy()

In [37]:
# Start a list for the pilot results
pilot_matches = []

for i, row in pilot_candidates.iterrows():

    search_title = row["search_title"]
    search_year = row["year"]

    try:
        result, score, search_method = search_tmdb_candidate(
            search_title,
            search_year
        )

        if result:
            pilot_matches.append({
                "imdb_title": row["movie"],
                "search_title": search_title,
                "imdb_year": search_year,
                "review_count": row["review_count"],
                "tmdb_id": result.get("id"),
                "media_type": result.get("media_type"),
                "tmdb_title": result.get("title") or result.get("name"),
                "tmdb_year": get_result_year(result),
                "match_score": score,
                "search_method": search_method
            })

        else:
            pilot_matches.append({
                "imdb_title": row["movie"],
                "search_title": search_title,
                "imdb_year": search_year,
                "review_count": row["review_count"],
                "tmdb_id": None,
                "media_type": None,
                "tmdb_title": None,
                "tmdb_year": None,
                "match_score": None,
                "search_method": search_method
            })

    except requests.RequestException as e:
        # Keep request errors so they can be reviewed or retried
        pilot_matches.append({
            "imdb_title": row["movie"],
            "search_title": search_title,
            "imdb_year": search_year,
            "review_count": row["review_count"],
            "tmdb_id": None,
            "media_type": None,
            "tmdb_title": None,
            "tmdb_year": None,
            "match_score": None,
            "search_method": "request_error"
        })

        print(f"Request failed for {search_title}: {e}")

    # Show progress every 10 titles
    if len(pilot_matches) % 10 == 0:
        print(f"{len(pilot_matches)} of {len(pilot_candidates)} processed")

10 of 100 processed
20 of 100 processed
30 of 100 processed
40 of 100 processed
50 of 100 processed
60 of 100 processed
70 of 100 processed
80 of 100 processed
90 of 100 processed
100 of 100 processed


In [38]:
# Build the pilot results table
pilot_matches_df = pd.DataFrame(pilot_matches)

print(pilot_matches_df.shape)
print(pilot_matches_df.columns.tolist())

(100, 10)
['imdb_title', 'search_title', 'imdb_year', 'review_count', 'tmdb_id', 'media_type', 'tmdb_title', 'tmdb_year', 'match_score', 'search_method']


In [40]:
# Review any matches that are still weak or unresolved
pilot_review = pilot_matches_df[
    (pilot_matches_df["match_score"] < 12)
    | (pilot_matches_df["match_score"].isna())
].copy()

pilot_review[
    [
        "imdb_title",
        "search_title",
        "imdb_year",
        "tmdb_title",
        "tmdb_year",
        "media_type",
        "match_score",
        "search_method"
    ]
]

,imdb_title,search_title,imdb_year,tmdb_title,tmdb_year,media_type,match_score,search_method
6,Star Wars: Episode VIII - The Last Jedi (2017),Star Wars: Episode VIII - The Last Jedi,2017,Star Wars: The Last Jedi,2017,movie,7.0,multi
7,小丑 (2019),小丑,2019,Joker,2019,movie,7.0,fallback
10,STAR WARS：天行者的崛起 (2019),STAR WARS：天行者的崛起,2019,Star Wars: The Rise of Skywalker,2019,movie,7.0,multi
12,Star Wars: Episode VII - The Force Awakens (2015),Star Wars: Episode VII - The Force Awakens,2015,Star Wars: The Force Awakens,2015,movie,7.0,multi
35,Asur: Welcome to Your Dark Side (2020– ),Asur: Welcome to Your Dark Side,2020,NaN,NaN,NaN,NaN,no_match
39,The Chosen (2017– ),The Chosen,2017,The Chosen Ones,2017,movie,7.0,fallback
58,獵魔士 (2019– ),獵魔士,2019,The Witcher,2019,tv,7.0,fallback
74,Birds of Prey (2020),Birds of Prey,2020,Birds of Prey (and the Fantabulous Emancipatio...,2020,movie,7.0,fallback
83,黑暗騎士：黎明昇起 (2012),黑暗騎士：黎明昇起,2012,The Dark Knight Rises,2012,movie,7.0,multi
94,V for Vendetta (2005),V for Vendetta,2005,V for Vendetta,2006,movie,10.0,fallback


In [42]:
# Convert the year columns to numeric values while keeping missing years as NaN
imdb_year_numeric = pd.to_numeric(
    pilot_matches_df["imdb_year"],
    errors="coerce"
)

tmdb_year_numeric = pd.to_numeric(
    pilot_matches_df["tmdb_year"],
    errors="coerce"
)

# Check for matched titles where the IMDb and TMDB years differ by more than one year
year_mismatches = pilot_matches_df[
    pilot_matches_df["tmdb_id"].notna()
    & tmdb_year_numeric.notna()
    & imdb_year_numeric.notna()
    & ((imdb_year_numeric - tmdb_year_numeric).abs() > 1)
]

print(f"Year mismatches greater than 1 year: {len(year_mismatches)}")

year_mismatches[
    [
        "imdb_title",
        "imdb_year",
        "tmdb_title",
        "tmdb_year",
        "media_type",
        "match_score",
        "search_method"
    ]
]

Year mismatches greater than 1 year: 0


,imdb_title,imdb_year,tmdb_title,tmdb_year,media_type,match_score,search_method


## 5. Full TMDB Candidate Matching

Apply the validated matching process to all IMDb candidates. Results are checkpointed every 100 titles so an interrupted run can resume without repeating completed searches.

In [7]:
# Create a folder for the TMDB ingestion results
tmdb_output_folder = project_root / "data" / "raw" / "tmdb"
tmdb_output_folder.mkdir(parents=True, exist_ok=True)

# This file will store our title-to-TMDB matching progress
match_output_file = tmdb_output_folder / "title_matches.parquet"

print("TMDB output folder:", tmdb_output_folder)
print("Match output file:", match_output_file)

TMDB output folder: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\raw\tmdb
Match output file: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\raw\tmdb\title_matches.parquet


In [8]:
# Load any matches from an earlier run so we do not repeat completed searches
if match_output_file.exists():
    existing_matches = pd.read_parquet(match_output_file)

    completed_titles = set(existing_matches["imdb_title"])

    print(f"Existing results loaded: {len(existing_matches):,}")
else:
    existing_matches = pd.DataFrame()
    completed_titles = set()

    print("No existing results found. Starting from the beginning.")

print(f"Titles already completed: {len(completed_titles):,}")

Existing results loaded: 8,841
Titles already completed: 8,841


In [9]:
# Start with any results that were already saved
all_matches = existing_matches.to_dict("records")

# Only process titles that have not already been completed
remaining_candidates = tmdb_candidates[
    ~tmdb_candidates["movie"].isin(completed_titles)
].copy()

print(f"Total candidates: {len(tmdb_candidates):,}")
print(f"Already completed: {len(completed_titles):,}")
print(f"Remaining: {len(remaining_candidates):,}")

Total candidates: 8,841
Already completed: 8,841
Remaining: 0


In [10]:
# Search TMDB for every remaining IMDb candidate
for processed, (_, row) in enumerate(
    remaining_candidates.iterrows(),
    start=1
):
    search_title = row["search_title"]
    search_year = row["year"]

    try:
        result, score, search_method = search_tmdb_candidate(
            search_title,
            search_year
        )

        match_status = assign_match_status(
            result,
            score,
            search_method
        )

        if result:
            match_record = {
                "imdb_title": row["movie"],
                "search_title": search_title,
                "imdb_year": search_year,
                "review_count": row["review_count"],
                "tmdb_id": result.get("id"),
                "media_type": result.get("media_type"),
                "tmdb_title": result.get("title") or result.get("name"),
                "tmdb_year": get_result_year(result),
                "match_score": score,
                "search_method": search_method,
                "match_status": match_status
            }

        else:
            match_record = {
                "imdb_title": row["movie"],
                "search_title": search_title,
                "imdb_year": search_year,
                "review_count": row["review_count"],
                "tmdb_id": None,
                "media_type": None,
                "tmdb_title": None,
                "tmdb_year": None,
                "match_score": None,
                "search_method": search_method,
                "match_status": match_status
            }

    except requests.RequestException as e:
        # Save request failures separately so they can be retried later
        match_record = {
            "imdb_title": row["movie"],
            "search_title": search_title,
            "imdb_year": search_year,
            "review_count": row["review_count"],
            "tmdb_id": None,
            "media_type": None,
            "tmdb_title": None,
            "tmdb_year": None,
            "match_score": None,
            "search_method": "request_error",
            "match_status": "request_error"
        }

        print(f"Request failed for {search_title}: {e}")

    all_matches.append(match_record)

    # Save progress every 100 titles so an interruption does not lose the run
    if processed % 100 == 0:
        pd.DataFrame(all_matches).to_parquet(
            match_output_file,
            index=False
        )

        print(
            f"{processed:,} of "
            f"{len(remaining_candidates):,} remaining titles processed"
        )

In [11]:
# Save again after the loop so the final partial batch is not lost
all_matches_df = pd.DataFrame(all_matches)

all_matches_df.to_parquet(
    match_output_file,
    index=False
)

print("\nTMDB candidate matching complete.")
print(f"Total results saved: {len(all_matches_df):,}")
print("Saved to:", match_output_file)


TMDB candidate matching complete.
Total results saved: 8,841
Saved to: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\raw\tmdb\title_matches.parquet


## 6. Validate Full Matching Results

Review match status, overall coverage, year inconsistencies, and the distribution of lower-confidence matches before selecting the records used downstream.

In [12]:
# Load the completed TMDB matching results
all_matches_df = pd.read_parquet(match_output_file)

print(f"Total results: {len(all_matches_df):,}")

print("\nMatch status counts:")
print(all_matches_df["match_status"].value_counts(dropna=False))

print("\nSearch method counts:")
print(all_matches_df["search_method"].value_counts(dropna=False))

Total results: 8,841

Match status counts:
match_status
high_confidence    7726
low_confidence      973
no_match            142
Name: count, dtype: int64

Search method counts:
search_method
multi       8196
fallback     503
no_match     142
Name: count, dtype: int64


In [13]:
# Count how many candidates received a TMDB ID
matched_count = all_matches_df["tmdb_id"].notna().sum()
unmatched_count = all_matches_df["tmdb_id"].isna().sum()

match_rate = matched_count / len(all_matches_df) * 100

print(f"Matched: {matched_count:,}")
print(f"Unmatched: {unmatched_count:,}")
print(f"Match rate: {match_rate:.1f}%")

Matched: 8,699
Unmatched: 142
Match rate: 98.4%


In [14]:
# Convert years safely so missing values do not cause errors
imdb_year_numeric = pd.to_numeric(
    all_matches_df["imdb_year"],
    errors="coerce"
)

tmdb_year_numeric = pd.to_numeric(
    all_matches_df["tmdb_year"],
    errors="coerce"
)

# Look for matched titles where the years differ by more than one
year_mismatches = all_matches_df[
    all_matches_df["tmdb_id"].notna()
    & imdb_year_numeric.notna()
    & tmdb_year_numeric.notna()
    & ((imdb_year_numeric - tmdb_year_numeric).abs() > 1)
]

print(f"Year mismatches greater than 1 year: {len(year_mismatches):,}")

Year mismatches greater than 1 year: 15


In [15]:
# Count the titles that need additional review
low_confidence = all_matches_df[
    all_matches_df["match_status"] == "low_confidence"
]

print(f"Low-confidence matches: {len(low_confidence):,}")

low_confidence.head(20)

Low-confidence matches: 973


,imdb_title,search_title,imdb_year,review_count,tmdb_id,media_type,tmdb_title,tmdb_year,match_score,search_method,match_status
6,Star Wars: Episode VIII - The Last Jedi (2017),Star Wars: Episode VIII - The Last Jedi,2017,6585,181808.0,movie,Star Wars: The Last Jedi,2017,7.0,multi,low_confidence
7,小丑 (2019),小丑,2019,6514,475557.0,movie,Joker,2019,7.0,fallback,low_confidence
10,STAR WARS：天行者的崛起 (2019),STAR WARS：天行者的崛起,2019,5147,181812.0,movie,Star Wars: The Rise of Skywalker,2019,7.0,multi,low_confidence
12,Star Wars: Episode VII - The Force Awakens (2015),Star Wars: Episode VII - The Force Awakens,2015,4782,140607.0,movie,Star Wars: The Force Awakens,2015,7.0,multi,low_confidence
39,The Chosen (2017– ),The Chosen,2017,3058,1553977.0,movie,The Chosen Ones,2017,7.0,fallback,low_confidence
58,獵魔士 (2019– ),獵魔士,2019,2744,71912.0,tv,The Witcher,2019,7.0,fallback,low_confidence
74,Birds of Prey (2020),Birds of Prey,2020,2390,495764.0,movie,Birds of Prey (and the Fantabulous Emancipatio...,2020,7.0,fallback,low_confidence
83,黑暗騎士：黎明昇起 (2012),黑暗騎士：黎明昇起,2012,2299,49026.0,movie,The Dark Knight Rises,2012,7.0,multi,low_confidence
94,V for Vendetta (2005),V for Vendetta,2005,2188,752.0,movie,V for Vendetta,2006,10.0,fallback,low_confidence
122,X-Men: Dark Phoenix (2019),X-Men: Dark Phoenix,2019,1949,320288.0,movie,Dark Phoenix,2019,7.0,fallback,low_confidence


In [16]:
# Look at the small group of matches with suspicious year differences
year_mismatches[
    [
        "imdb_title",
        "search_title",
        "imdb_year",
        "tmdb_title",
        "tmdb_year",
        "media_type",
        "match_score",
        "search_method",
        "match_status"
    ]
]

,imdb_title,search_title,imdb_year,tmdb_title,tmdb_year,media_type,match_score,search_method,match_status
714,The Upside (2017),The Upside,2017,The Upside,2019,movie,10.0,fallback,low_confidence
1205,Hamilton (2020),Hamilton,2020,Hamilton,2025,movie,10.0,fallback,low_confidence
1822,Terrifier (2016),Terrifier,2016,Terrifier,2018,movie,10.0,fallback,low_confidence
2268,You're Next (2011),You're Next,2011,You're Next,2013,movie,10.0,fallback,low_confidence
4063,The Blackcoat's Daughter (I) (2015),The Blackcoat's Daughter,2015,The Blackcoat's Daughter,2017,movie,10.0,fallback,low_confidence
5439,Miss Lovely (2012),Miss Lovely,2012,Miss Lovely,2014,movie,10.0,fallback,low_confidence
6092,Begotten (1989),Begotten,1989,Begotten,1991,movie,10.0,fallback,low_confidence
6130,Fish Out of Water (2005),Fish Out of Water,2005,Fish Out of Water,2008,movie,10.0,fallback,low_confidence
6862,Cannibal! The Musical (1993),Cannibal! The Musical,1993,Cannibal! The Musical,1996,movie,10.0,fallback,low_confidence
7128,The War of the Worlds (2005 Video),The War of the Worlds,2005,The War of the Worlds,1953,movie,9.8,fallback,low_confidence


In [17]:
# See how the low-confidence matches are distributed across scores
low_confidence["match_score"].value_counts().sort_index()

match_score
2.0       9
6.6       1
6.8      18
7.0     537
9.4       1
9.6       1
9.8       7
10.0    287
10.2      2
10.4      3
10.8      3
11.0      4
11.2      5
11.4      9
11.6     26
11.8     60
Name: count, dtype: int64

### Matching Summary

- 8,841 IMDb candidate titles were evaluated.
- 8,699 received a TMDB match, for a 98.4% match rate.
- 7,726 matches met the high-confidence threshold and are used downstream.
- 973 lower-confidence matches were retained for auditing but excluded from downstream analysis.
- 142 titles received no TMDB match.
- 15 matched titles differed from the IMDb year by more than one year; all were lower-confidence matches.

## 7. Create Trusted IMDb–TMDB Crosswalk

Only high-confidence matches are used in downstream analysis. Lower-confidence and unmatched records remain in the complete matching output for auditing rather than being manually forced into a match.

From 8,841 IMDb candidates, 7,726 matches met the high-confidence threshold.

In [18]:
# Keep only the matches we trust for downstream use
high_confidence_matches = all_matches_df[
    all_matches_df["match_status"] == "high_confidence"
].copy()

# Save the trusted matches separately
high_confidence_output_file = (
    project_root
    / "data"
    / "processed"
    / "high_confidence_tmdb_matches.parquet"
)

high_confidence_matches.to_parquet(
    high_confidence_output_file,
    index=False
)

print(f"High-confidence matches saved: {len(high_confidence_matches):,}")
print("Saved to:", high_confidence_output_file)

High-confidence matches saved: 7,726
Saved to: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\high_confidence_tmdb_matches.parquet


## 8. Retrieve Detailed TMDB Metadata

TMDB search results contain enough information for entity matching but not all fields needed for downstream analysis. Detailed movie and TV endpoints are queried for the trusted matches to retrieve genres, runtime, overview, language, production countries, ratings, and TV-specific fields.

In [19]:
# Fetch the detailed TMDB metadata for one trusted match
def fetch_tmdb_details(tmdb_id, media_type):
    # Movies and TV shows use different detail endpoints
    endpoint = (
        f"/movie/{int(tmdb_id)}"
        if media_type == "movie"
        else f"/tv/{int(tmdb_id)}"
    )

    response = requests.get(
        f"{tmdb_base_url}{endpoint}",
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

### Verify Movie and TV Detail Responses

Check one record of each media type because TMDB uses different field names and metadata structures for movies and TV shows.

In [18]:
# Test one movie
movie_test = high_confidence_matches[
    high_confidence_matches["media_type"] == "movie"
].iloc[0]

movie_details = fetch_tmdb_details(
    movie_test["tmdb_id"],
    movie_test["media_type"]
)

print("MOVIE")
print("Title:", movie_details.get("title"))
print("Genres:", movie_details.get("genres"))
print("Runtime:", movie_details.get("runtime"))
print("Release date:", movie_details.get("release_date"))

MOVIE
Title: Avengers: Endgame
Genres: [{'id': 12, 'name': 'Adventure'}, {'id': 878, 'name': 'Science Fiction'}, {'id': 28, 'name': 'Action'}]
Runtime: 181
Release date: 2019-04-24


In [19]:
# Test one TV show
tv_test = high_confidence_matches[
    high_confidence_matches["media_type"] == "tv"
].iloc[0]

tv_details = fetch_tmdb_details(
    tv_test["tmdb_id"],
    tv_test["media_type"]
)

print("TV SHOW")
print("Title:", tv_details.get("name"))
print("Genres:", tv_details.get("genres"))
print("Episode runtime:", tv_details.get("episode_run_time"))
print("First air date:", tv_details.get("first_air_date"))
print("Seasons:", tv_details.get("number_of_seasons"))

TV SHOW
Title: Game of Thrones
Genres: [{'id': 10765, 'name': 'Sci-Fi & Fantasy'}, {'id': 18, 'name': 'Drama'}, {'id': 10759, 'name': 'Action & Adventure'}]
Episode runtime: []
First air date: 2011-04-17
Seasons: 8


In [20]:
# Create a file for the detailed TMDB metadata
tmdb_details_file = (
    project_root
    / "data"
    / "raw"
    / "tmdb"
    / "title_details.parquet"
)

# Load any previously saved metadata so the run can resume if interrupted
if tmdb_details_file.exists():
    existing_details = pd.read_parquet(tmdb_details_file)

    # A TMDB entity is identified by both its ID and media type
    completed_tmdb_records = set(
        zip(
            existing_details["tmdb_id"].astype(int),
            existing_details["media_type"]
        )
    )

    print(f"Existing metadata loaded: {len(existing_details):,}")
else:
    existing_details = pd.DataFrame()
    completed_tmdb_records = set()

    print("No existing metadata found. Starting from the beginning.")

print(
    f"Unique TMDB entities already completed: "
    f"{len(completed_tmdb_records):,}"
)

Existing metadata loaded: 7,726
Unique TMDB entities already completed: 7,721


In [21]:
# Identify which trusted match rows still need metadata
match_keys = list(
    zip(
        high_confidence_matches["tmdb_id"].astype(int),
        high_confidence_matches["media_type"]
    )
)

remaining_mask = [
    key not in completed_tmdb_records
    for key in match_keys
]

remaining_matches = high_confidence_matches[
    remaining_mask
].copy()

print(f"Total trusted matches: {len(high_confidence_matches):,}")
print(f"Remaining: {len(remaining_matches):,}")

Total trusted matches: 7,726
Remaining: 0


In [22]:
# Start with any metadata that was already saved
all_details = existing_details.to_dict("records")

# Fetch the detailed metadata for every remaining trusted TMDB match
for processed, (_, row) in enumerate(
    remaining_matches.iterrows(),
    start=1
):
    tmdb_id = int(row["tmdb_id"])
    media_type = row["media_type"]

    try:
        details = fetch_tmdb_details(
            tmdb_id,
            media_type
        )

        # Movies and TV shows use different field names for some metadata
        if media_type == "movie":
            title = details.get("title")
            original_title = details.get("original_title")
            release_date = details.get("release_date")
            runtime = details.get("runtime")
            number_of_seasons = None
            number_of_episodes = None
        else:
            title = details.get("name")
            original_title = details.get("original_name")
            release_date = details.get("first_air_date")

            episode_runtime = details.get("episode_run_time", [])

            # Use the first listed runtime when TMDB provides one
            runtime = (
                episode_runtime[0]
                if episode_runtime
                else None
            )

            number_of_seasons = details.get("number_of_seasons")
            number_of_episodes = details.get("number_of_episodes")

        detail_record = {
            "tmdb_id": tmdb_id,
            "media_type": media_type,
            "title": title,
            "original_title": original_title,
            "overview": details.get("overview"),

            # Keep only the genre names instead of the full TMDB objects
            "genres": [
                genre["name"]
                for genre in details.get("genres", [])
            ],

            "original_language": details.get("original_language"),
            "release_date": release_date,
            "runtime": runtime,
            "status": details.get("status"),
            "popularity": details.get("popularity"),
            "vote_average": details.get("vote_average"),
            "vote_count": details.get("vote_count"),

            # Keep only the country codes for a simpler structure
            "production_countries": [
                country["iso_3166_1"]
                for country in details.get("production_countries", [])
            ],

            "number_of_seasons": number_of_seasons,
            "number_of_episodes": number_of_episodes,
            "metadata_status": "success"
        }

    except requests.RequestException as e:
        # Keep failed requests so we know which IDs may need to be retried
        detail_record = {
            "tmdb_id": tmdb_id,
            "media_type": media_type,
            "title": None,
            "original_title": None,
            "overview": None,
            "genres": None,
            "original_language": None,
            "release_date": None,
            "runtime": None,
            "status": None,
            "popularity": None,
            "vote_average": None,
            "vote_count": None,
            "production_countries": None,
            "number_of_seasons": None,
            "number_of_episodes": None,
            "metadata_status": "request_error"
        }

        print(f"Request failed for TMDB ID {tmdb_id}: {e}")

    all_details.append(detail_record)

    # Save progress every 100 titles so an interruption does not lose the run
    if processed % 100 == 0:
        pd.DataFrame(all_details).to_parquet(
            tmdb_details_file,
            index=False
        )

        print(
            f"{processed:,} of "
            f"{len(remaining_matches):,} remaining titles processed"
        )

# Save the final partial batch
all_details_df = pd.DataFrame(all_details)

all_details_df.to_parquet(
    tmdb_details_file,
    index=False
)

print("\nTMDB metadata ingestion complete.")
print(f"Total records saved: {len(all_details_df):,}")
print("Saved to:", tmdb_details_file)


TMDB metadata ingestion complete.
Total records saved: 7,726
Saved to: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\raw\tmdb\title_details.parquet


## 9. Validate Retrieved Metadata

Confirm that the metadata requests succeeded, inspect missing values, and check entity uniqueness before creating the final processed metadata table.

In [23]:
# Check whether every metadata request succeeded
print("Metadata status:")
print(all_details_df["metadata_status"].value_counts(dropna=False))

# Confirm the expected number of records
print(f"\nTotal records: {len(all_details_df):,}")

Metadata status:
metadata_status
success    7726
Name: count, dtype: int64

Total records: 7,726


In [24]:
# Check missing values in the main metadata fields
fields_to_check = [
    "title",
    "overview",
    "genres",
    "original_language",
    "release_date",
    "runtime",
    "status",
    "vote_average",
    "vote_count"
]

print("Missing values:")
print(
    all_details_df[fields_to_check]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Missing values:
runtime              419
title                  0
overview               0
original_language      0
genres                 0
release_date           0
status                 0
vote_average           0
vote_count             0
dtype: int64


### Duplicate Entity Check

TMDB movie and TV records use separate ID namespaces, so uniqueness is evaluated using the composite key `(tmdb_id, media_type)` rather than `tmdb_id` alone.

In [25]:
# Check for true duplicates using the TMDB record's actual unique key
duplicate_count = all_details_df.duplicated(
    subset=["tmdb_id", "media_type"]
).sum()

print("Duplicate TMDB records:", duplicate_count)

Duplicate TMDB records: 5


In [26]:
# Find the true duplicate TMDB records
true_duplicates = all_details_df[
    all_details_df.duplicated(
        subset=["tmdb_id", "media_type"],
        keep=False
    )
].sort_values(["media_type", "tmdb_id"])

true_duplicates[
    [
        "tmdb_id",
        "media_type",
        "title",
        "release_date"
    ]
]

,tmdb_id,media_type,title,release_date
46,74,movie,War of the Worlds,2005-06-28
7145,74,movie,War of the Worlds,2005-06-28
38,466272,movie,Once Upon a Time... in Hollywood,2019-07-24
155,466272,movie,Once Upon a Time... in Hollywood,2019-07-24
3011,4194,tv,Star Wars: The Clone Wars,2008-10-03
3292,4194,tv,Star Wars: The Clone Wars,2008-10-03
4207,86430,tv,Your Honor,2020-12-06
5363,86430,tv,Your Honor,2020-12-06
6585,89456,tv,Primal,2019-10-08
7204,89456,tv,Primal,2019-10-08


In [27]:
# See which IMDb titles produced the duplicate TMDB records
duplicate_keys = true_duplicates[
    ["tmdb_id", "media_type"]
].drop_duplicates()

duplicate_match_sources = high_confidence_matches.merge(
    duplicate_keys,
    on=["tmdb_id", "media_type"],
    how="inner"
)

duplicate_match_sources[
    [
        "imdb_title",
        "tmdb_id",
        "media_type",
        "tmdb_title",
        "match_score"
    ]
].sort_values(["media_type", "tmdb_id"])

,imdb_title,tmdb_id,media_type,tmdb_title,match_score
1,War of the Worlds (2005),74.0,movie,War of the Worlds,14.8
8,War of the Worlds (2005 Video),74.0,movie,War of the Worlds,14.8
0,Once Upon a Time... In Hollywood (2019),466272.0,movie,Once Upon a Time... in Hollywood,15.0
2,Once Upon a Time... in Hollywood (2019),466272.0,movie,Once Upon a Time... in Hollywood,15.0
3,Star Wars: The Clone Wars (2008),4194.0,tv,Star Wars: The Clone Wars,15.0
4,Star Wars: The Clone Wars (2008–2020),4194.0,tv,Star Wars: The Clone Wars,15.0
5,Your Honor (II) (2020– ),86430.0,tv,Your Honor,15.0
6,Your Honor (I) (2020– ),86430.0,tv,Your Honor,15.0
7,Primal (2019),89456.0,tv,Primal,15.0
9,Primal (2019– ),89456.0,tv,Primal,15.0


## 10. Create Clean TMDB Metadata Table

The raw ingestion output is preserved unchanged. For downstream analysis, repeated TMDB entities are collapsed so the processed metadata table contains one row per unique movie or TV entity.

In [28]:
# Keep one metadata record for each unique TMDB movie or TV show
tmdb_metadata = all_details_df.drop_duplicates(
    subset=["tmdb_id", "media_type"]
).copy()

print(f"Before: {len(all_details_df):,}")
print(f"After: {len(tmdb_metadata):,}")

print(
    "Remaining duplicate TMDB records:",
    tmdb_metadata.duplicated(
        subset=["tmdb_id", "media_type"]
    ).sum()
)

Before: 7,726
After: 7,721
Remaining duplicate TMDB records: 0


The five repeated metadata rows came from multiple IMDb title strings mapping to the same TMDB entity. The IMDb-to-TMDB crosswalk retains those source mappings, while the processed metadata table keeps one row per unique TMDB movie or TV entity.

In [29]:
processed_metadata_file = (
    project_root
    / "data"
    / "processed"
    / "tmdb_title_metadata.parquet"
)

# Save one clean metadata record per unique TMDB movie or TV show
tmdb_metadata.to_parquet(
    processed_metadata_file,
    index=False
)

print(f"Records saved: {len(tmdb_metadata):,}")
print("Saved to:", processed_metadata_file)

Records saved: 7,721
Saved to: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\processed\tmdb_title_metadata.parquet


## Output

This notebook produces three TMDB datasets:

- `data/raw/tmdb/title_matches.parquet` — complete IMDb-to-TMDB matching results, including high-confidence, low-confidence, and unmatched candidates.
- `data/processed/high_confidence_tmdb_matches.parquet` — trusted IMDb-to-TMDB crosswalk used for downstream analysis.
- `data/processed/tmdb_title_metadata.parquet` — cleaned TMDB metadata with one record per unique movie or TV entity.

The next stage uses the trusted crosswalk to connect IMDb audience reviews to these TMDB entities.